# 🧪 W3-D1 概念实验：Scaling Law 与涌现能力

> 配套阅读：`ima/第3周-Day1-预训练数据规模与涌现能力.md`
>
> 用可执行代码回答：
> 1. **Scaling Law**：参数量翻倍，Loss 能降多少？
> 2. **涌现能力**：为什么小模型不会算术，大模型突然会了？

## 实验 1：Scaling Law — Loss 与参数量的幂律关系

$L(N) \approx A \cdot N^{-\alpha} + L_\infty$，$\alpha \approx 0.076$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
params = np.logspace(7.5, 12.5, 200)
alpha_val = 0.076; A = 6.0; L_inf = 0.5
loss = A * (params / 1e12) ** (-alpha_val) + L_inf
milestones = [(1e9,'1B\nGPT-2'),(7e9,'7B\nQwen2'),(7e10,'70B\nLLaMA'),(1.75e11,'175B\nGPT-3'),(1e12,'1T')]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.loglog(params, loss, 'b-', lw=2, label='Loss = A·N^(-α)+L∞, α=0.076')
for n, label in milestones:
    l = A*(n/1e12)**(-alpha_val)+L_inf
    ax.plot(n, l, 'ro', ms=8)
    ax.annotate(label, (n, l), textcoords='offset points', xytext=(8,10), fontsize=9, fontweight='bold')
ax.set_xlabel('参数量 (N)'); ax.set_ylabel('验证集 Loss')
ax.set_title('Scaling Law：参数量越大 Loss 越低，但收益递减')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3, which='both'); plt.tight_layout(); plt.show()
print(f"  1B → 175B: Loss {A*(1e9/1e12)**(-alpha_val)+L_inf:.3f} → {A*(1.75e11/1e12)**(-alpha_val)+L_inf:.3f}")
print(f"  175B → 1T:  Loss {A*(1.75e11/1e12)**(-alpha_val)+L_inf:.3f} → {A*(1e12/1e12)**(-alpha_val)+L_inf:.3f}")
print("→ 同样翻倍，前半段降幅远大于后半段：边际递减。")

## 实验 2：涌现的「相变」曲线

用 Sigmoid 模拟三种任务在规模阈值附近的相变。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
sizes = np.logspace(8, 12, 300)
tasks = {'简单加法':(1e9,0.5,'#e74c3c',0.15),'代码补全':(5e9,0.7,'#2ecc71',0.20),'多步推理':(3e10,1.2,'#3498db',0.25)}
fig, ax = plt.subplots(figsize=(10, 6))
for name,(thr,steep,col,noise) in tasks.items():
    acc = 1/(1+np.exp(-steep*(np.log10(sizes)-np.log10(thr))))
    acc += np.random.normal(0,noise,len(sizes)); acc = np.clip(acc,0,1)
    ax.semilogx(sizes, acc, '-', color=col, lw=2, label=f'{name}(阈值≈{thr:.0e})')
ax.axhspan(0,0.15,alpha=0.08,color='red'); ax.text(2e11,0.07,'「完全不会」区',fontsize=10,color='red',ha='center')
ax.set_xlabel('模型参数量'); ax.set_ylabel('任务准确率')
ax.set_title('涌现 ≈ Sigmoid 相变：不是线性增长，是「突然开窍」')
ax.legend(fontsize=11); ax.set_ylim(-0.05,1.05); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()
print("每个能力有自己的涌现阈值；跨过前几乎为零，跨过后快速上升。")

## 实验 3：预训练 Loss 下降曲线

10000 步预训练：初期暴跌、后期趋平。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
np.random.seed(0); steps = np.arange(1,10001)
loss = 4.0*np.exp(-steps/2000)+0.6+0.015*np.random.randn(len(steps))
fig,(a1,a2)=plt.subplots(1,2,figsize=(13,5))
a1.plot(steps,loss,'b-',lw=0.7,alpha=0.8); a1.set_xlabel('训练步数'); a1.set_ylabel('Loss')
a1.set_title('线性坐标：前2000步暴跌'); a1.grid(True,alpha=0.3)
a2.plot(steps,loss,'b-',lw=0.7,alpha=0.8); a2.set_yscale('log')
a2.set_xlabel('训练步数'); a2.set_ylabel('Loss(log)')
a2.set_title('对数坐标：全程在降，斜率越来越缓'); a2.grid(True,alpha=0.3,which='both')
plt.tight_layout(); plt.show()
print("Loss 从~4.6降到~0.6，但5000步后每1000步降幅极小。")

## 结论

| 问题 | 实验证据 |
|---|---|
| Scaling Law | 实验1：幂律曲线，边际递减 |
| 涌现形状 | 实验2：Sigmoid 相变 |
| 预训练为什么贵 | 实验3：后期 Loss 下降极慢 |

→ 深入阅读：`ima/第3周-Day1-预训练数据规模与涌现能力.md`